# Train YOLO26s — v2 (Improved Augmentation + Training Schedule)

Implements Priority 1C + 2E + 2F recommendations from `training_report.md`.

**Changes vs v1:**
| Param | v1 | v2 | Reason |
|---|---|---|---|
| `copy_paste` | 0.3 | **0.6** | More rare-class instances (Actinomyces, TV) pasted into images |
| `mixup` | 0.0 | **0.15** | Class boundary regularisation on small dataset |
| `multi_scale` | 0.0 | **0.5** | Trains at ±50% imgsz — helps Candida size variance |
| `cos_lr` | False | **True** | Cosine LR avoids plateau early stopping too soon |
| `lrf` | 0.01 | **0.001** | Lower LR floor gives smoother cosine tail |
| `epochs` | 150 | **200** | Extended window for cosine schedule to utilise |
| `patience` | 30 | **50** | Match longer training; less hair-trigger early stop |

Outputs: `checkpoints/yolo26s_v2/` · `results/metrics/yolo26s_v2_metrics.csv` · `results/plots/yolo26s_v2/`

| Split | mAP@50 | mAP@50-95 | Precision | Recall |
|---|---|---|---|---|
| Val  | — | — | — | — |
| Test | — | — | — | — |

| Class | Val AP@50 | Test AP@50 |
|---|---|---|
| Trichomonas vaginalis      | — | — |
| Bacterial vaginosis        | — | — |
| Candida spp.               | — | — |
| Actinomyces spp.           | — | — |

In [ ]:
import os
import tempfile

os.makedirs('/dev/shm/t', exist_ok=True)
os.environ['TMPDIR'] = '/dev/shm/t'
tempfile.tempdir = '/dev/shm/t'

In [ ]:
MODEL_VARIANT = 'yolo26s'
VERSION       = 'v2'
RUN_NAME      = f'{MODEL_VARIANT}_{VERSION}'

# --- v2 training schedule ---
BATCH         = 8
EPOCHS        = 200     # v1: 150
IMGSZ         = 1280
WORKERS       = 4
PATIENCE      = 50      # v1: 30
LRF           = 0.001   # v1: 0.01 (default)

from pathlib import Path
ROOT           = Path('../../../').resolve()
DATASET_YAML   = ROOT / 'models/organism_det/configs/dataset.yaml'
CHECKPOINT_DIR = ROOT / 'models/organism_det/checkpoints' / RUN_NAME
METRICS_DIR    = ROOT / 'models/organism_det/results/metrics'
PLOTS_DIR      = ROOT / 'models/organism_det/results/plots' / RUN_NAME

for d in [CHECKPOINT_DIR, METRICS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert DATASET_YAML.exists(), f'Run 00_data_converter.ipynb first. Missing: {DATASET_YAML}'

In [ ]:
import subprocess, sys

for pkg in ['ultralytics', 'pandas', 'matplotlib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import ultralytics
print(f'ultralytics {ultralytics.__version__}')
print(f'torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
from ultralytics import YOLO

model = YOLO(f'{MODEL_VARIANT}.pt')

model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    patience=PATIENCE,
    project=str(CHECKPOINT_DIR.parent),
    name=RUN_NAME,
    exist_ok=True,
    optimizer='auto',
    # --- v2: augmentation changes (Priority 1C) ---
    mosaic=1.0,
    close_mosaic=10,
    copy_paste=0.6,     # v1: 0.3 — more rare-class pasting
    mixup=0.15,         # v1: 0.0 — class boundary regularisation
    # --- v2: multi-scale training (Priority 2F) ---
    multi_scale=0.5,    # v1: 0.0 — ±50% imgsz, helps Candida size variance
    # --- v2: LR schedule (Priority 2E) ---
    cos_lr=True,        # v1: False
    lrf=LRF,            # v1: 0.01
    # --- unchanged augmentations ---
    flipud=0.5,
    fliplr=0.5,
    degrees=15.0,
    hsv_h=0.05,
    hsv_s=0.3,
    hsv_v=0.1,
    scale=0.5,
    plots=True,
    save=True,
    save_period=50,
    verbose=True,
)

### Copy weights and training plots to results directories

In [ ]:
import shutil

RUN_DIR = CHECKPOINT_DIR.parent / RUN_NAME

for weight in ['best.pt', 'last.pt']:
    src = RUN_DIR / 'weights' / weight
    if src.exists():
        shutil.copy(src, CHECKPOINT_DIR / weight)

for fname in [
    'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'PR_curve.png', 'F1_curve.png', 'results.png', 'labels.jpg',
]:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, PLOTS_DIR / fname)

print(f'Weights → {CHECKPOINT_DIR}')
print(f'Plots   → {PLOTS_DIR}')

### Evaluate on val and test sets

In [ ]:
import pandas as pd
from ultralytics import YOLO as _YOLO

CLASS_NAMES = [
    'Trichomonas vaginalis',
    'Bacterial vaginosis flora shift',
    'Candida spp.',
    'Actinomyces spp.',
]

best_model = _YOLO(str(CHECKPOINT_DIR / 'best.pt'))

rows = []
for split in ['val', 'test']:
    r = best_model.val(data=str(DATASET_YAML), split=split, imgsz=IMGSZ, verbose=False)
    row = {
        'model':     RUN_NAME,
        'split':     split,
        'mAP50':     round(float(r.box.map50), 4),
        'mAP50_95':  round(float(r.box.map),   4),
        'precision': round(float(r.box.mp),     4),
        'recall':    round(float(r.box.mr),     4),
    }
    for i, name in enumerate(CLASS_NAMES):
        key = f'AP50_{name.replace(" ", "_")}'
        row[key] = round(float(r.box.ap50[i]), 4) if i < len(r.box.ap50) else None
    rows.append(row)

metrics_df = pd.DataFrame(rows)
out_path   = METRICS_DIR / f'{RUN_NAME}_metrics.csv'
metrics_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}\n')
print(metrics_df.to_string(index=False))

### Sample predictions on validation images

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CLASS_COLORS = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
val_images   = sorted((ROOT / 'data/organisms_data/val/images').glob('*.jpg'))
samples      = random.sample(val_images, min(8, len(val_images)))

fig, axes = plt.subplots(2, 4, figsize=(32, 14))
for ax, img_path in zip(axes.flat, samples):
    img   = Image.open(img_path)
    preds = best_model.predict(str(img_path), imgsz=IMGSZ, verbose=False)[0]
    ax.imshow(img)
    for box in preds.boxes:
        cls  = int(box.cls)
        conf = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor='none'
        )
        ax.add_patch(rect)
        label = f'{CLASS_NAMES[cls][:14]} {conf:.2f}'
        ax.text(x1, y1 - 4, label, color=CLASS_COLORS[cls], fontsize=6, fontweight='bold')
    ax.axis('off')
    ax.set_title(img_path.stem, fontsize=8)

fig.suptitle(f'{RUN_NAME.upper()} — Validation Sample Predictions', fontsize=14)
plt.tight_layout()
out_path = PLOTS_DIR / 'sample_predictions.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

### Results summary

In [ ]:
val_row  = metrics_df[metrics_df.split == 'val'].iloc[0]
test_row = metrics_df[metrics_df.split == 'test'].iloc[0]

print(f'\n{"=" * 62}')
print(f'  {RUN_NAME.upper()}  RESULTS')
print(f'{"=" * 62}')
print(f'  {"Metric":<24} {"Val":>8} {"Test":>8}')
print(f'  {"-" * 42}')
for metric, label in [("mAP50", "mAP@50"), ("mAP50_95", "mAP@50-95"),
                       ("precision", "Precision"), ("recall", "Recall")]:
    print(f'  {label:<24} {val_row[metric]:>8.4f} {test_row[metric]:>8.4f}')
print(f'\n  Per-class AP@50 (test):')
for name in CLASS_NAMES:
    key    = f'AP50_{name.replace(" ", "_")}'
    val    = test_row.get(key)
    suffix = '  <- critical' if 'Trichomonas' in name else ''
    v_str  = f'{val:.4f}' if val is not None else 'N/A'
    print(f'    {name:<42} {v_str}{suffix}')
print(f'{"=" * 62}')